In [3]:
import os
import glob
import pandas as pd
import numpy as np
import cv2
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from tqdm.notebook import tqdm_notebook
import re
tqdm_notebook.pandas()
import tensorflow as tf
from tensorflow import keras
from keras.models import load_model
from keras.models import Sequential
from keras.layers import Dense, Activation
from keras.layers import Dropout
from keras.layers import Flatten
from tensorflow.keras.layers import Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from keras.preprocessing.image import ImageDataGenerator 
from sklearn import metrics
import itertools
import pickle
from joblib import dump, load
from PIL import Image


In [2]:
df_train = pd.read_csv('C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\save fichier rakuten\\df_train_cleaned.csv')

df_train= df_train.drop("Unnamed: 0", axis=1)

df_train.head()

,designation,description,productid,imageid,nom_img,Texte,prdtypecode,Catégories
0,Olivia: Personalisiertes Notizbuch / 150 Seite...,NaN,3804725264,1263597046,image_1263597046_product_3804725264,olivia personalisiertes notizbuch seiten punkt...,10,Livres occasions
1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,NaN,436067568,1008141237,image_1008141237_product_436067568,journal arts art marche salon art asiatique pa...,2280,"Journaux, revues d'occasion"
2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,image_938777978_product_201115110,grand stylet ergonomique bleu gamepad nintendo...,50,accessories gaming
3,Peluche Donald - Europe - Disneyland 2000 (Mar...,NaN,50418756,457047496,image_457047496_product_50418756,peluche donald europe disneyland marionnette d...,1280,Jouets petite enfance
4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,image_1077757786_product_278535884,guerre tuques luc grandeur veut organiser jeu ...,2705,Livres neufs


In [3]:
# L'emplacement que vous souhaitez ajouter
emplacement = 'C:\\Users\\yahya\\OneDrive\Documents\\Rakuten\\images\\images\\image_resized\\'

# Fonction pour ajouter l'emplacement au nom de l'image
def ajouter_emplacement(nom_img):
    return emplacement + nom_img + ".jpg"

# Appliquer la fonction à la colonne 'nom_img'
df_train['nom_img'] = df_train['nom_img'].apply(ajouter_emplacement)

df_train.head()

,designation,description,productid,imageid,nom_img,Texte,prdtypecode,Catégories
0,Olivia: Personalisiertes Notizbuch / 150 Seite...,NaN,3804725264,1263597046,C:\Users\yahya\OneDrive\Documents\Rakuten\imag...,olivia personalisiertes notizbuch seiten punkt...,10,Livres occasions
1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,NaN,436067568,1008141237,C:\Users\yahya\OneDrive\Documents\Rakuten\imag...,journal arts art marche salon art asiatique pa...,2280,"Journaux, revues d'occasion"
2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,C:\Users\yahya\OneDrive\Documents\Rakuten\imag...,grand stylet ergonomique bleu gamepad nintendo...,50,accessories gaming
3,Peluche Donald - Europe - Disneyland 2000 (Mar...,NaN,50418756,457047496,C:\Users\yahya\OneDrive\Documents\Rakuten\imag...,peluche donald europe disneyland marionnette d...,1280,Jouets petite enfance
4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,C:\Users\yahya\OneDrive\Documents\Rakuten\imag...,guerre tuques luc grandeur veut organiser jeu ...,2705,Livres neufs


In [4]:
# Ajuster les options d'affichage des colonnes
pd.set_option('display.max_colwidth', None)

# Afficher le contenu complet de la colonne 'nom_img' pour la première ligne
print(df_train['nom_img'].head())


0    C:\Users\yahya\OneDrive\Documents\Rakuten\images\images\image_resized\image_1263597046_product_3804725264.jpg
1     C:\Users\yahya\OneDrive\Documents\Rakuten\images\images\image_resized\image_1008141237_product_436067568.jpg
2      C:\Users\yahya\OneDrive\Documents\Rakuten\images\images\image_resized\image_938777978_product_201115110.jpg
3       C:\Users\yahya\OneDrive\Documents\Rakuten\images\images\image_resized\image_457047496_product_50418756.jpg
4     C:\Users\yahya\OneDrive\Documents\Rakuten\images\images\image_resized\image_1077757786_product_278535884.jpg
Name: nom_img, dtype: object


In [5]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84916 entries, 0 to 84915
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   designation  84916 non-null  object
 1   description  55116 non-null  object
 2   productid    84916 non-null  int64 
 3   imageid      84916 non-null  int64 
 4   nom_img      84916 non-null  object
 5   Texte        84906 non-null  object
 6   prdtypecode  84916 non-null  int64 
 7   Catégories   84916 non-null  object
dtypes: int64(3), object(5)
memory usage: 5.2+ MB


In [6]:
# Sauvegarder la data 
df_train.to_csv('C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\save fichier rakuten\\df_train_image.csv')

# Architecture LeNet

In [7]:
df_train_img = pd.read_csv('C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\save fichier rakuten\\df_train_image.csv')

df_train_img= df_train_img.drop("Unnamed: 0", axis=1)


In [8]:
df_train_img.shape

(84916, 8)

In [9]:
X_train_img, X_test_img = train_test_split(df_train_img, train_size=0.8)

# Data generateur

In [10]:
img_rows = 250
img_cols = 250
batch_size = 64
epochs = 100
emplacement = 'C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\images\\images\\image_resized'

In [11]:
#Générateur de données
datagen = ImageDataGenerator(rotation_range = 10,
                                width_shift_range = 0.1, # Translation horizontale
                                height_shift_range= 0.1, # Translation verticale
                                zoom_range = 0.1,
                                horizontal_flip=True,
                                validation_split = 0.2)

In [12]:
X_train_img["prdtypecode"] = X_train_img["prdtypecode"].astype(str)
X_test_img["prdtypecode"] = X_test_img["prdtypecode"].astype(str)

In [13]:
#Itérateur 
train_generator = datagen.flow_from_dataframe(dataframe = X_train_img,                                           
                                            x_col = "nom_img",
                                            y_col = 'prdtypecode',  #target data                                            
                                            class_mode = 'sparse',
                                            subset = 'training',
                                            # target_size = (256 , 256) - default
                                            #color_mode = 'rgb'- default
                                            batch_size = 32 # default
                                           )

test_generator = datagen.flow_from_dataframe(dataframe = X_test_img,                                            
                                            x_col = "nom_img",
                                            y_col ='prdtypecode', 
                                            class_mode = 'sparse',
                                            subset = 'validation', 
                                            #target_size = (256 , 256)-default
                                            #color_mode = 'rgb'- default
                                            batch_size = 32 # default
                                   )

Found 54346 validated image filenames belonging to 27 classes.
Found 3396 validated image filenames belonging to 27 classes.


In [4]:
# Initialisation du modèle
lenet = Sequential()

# Première couche de convolution
conv_1 = Conv2D(filters=30, kernel_size=(5, 5), padding='valid', input_shape=(256, 256, 3), activation='relu')
max_pool_1 = MaxPooling2D(pool_size=(2, 2))

# Deuxième couche de convolution
conv_2 = Conv2D(filters=16, kernel_size=(3, 3), padding='valid', activation='relu')
max_pool_2 = MaxPooling2D(pool_size=(2, 2))

# Couche d'aplatissement (Flatten)
flatten = Flatten()

# Couche de régularisation (Dropout)
dropout = Dropout(rate=0.2)

# Première couche dense
dense_1 = Dense(units=128, activation='relu')

# Couche de sortie (deuxième couche dense)
dense_2 = Dense(units=27, activation='softmax')  # Assurez-vous que le nombre d'unités correspond au nombre de classes

# Ajout des couches au modèle
lenet.add(conv_1)
lenet.add(max_pool_1)
lenet.add(conv_2)
lenet.add(max_pool_2)

lenet.add(dropout)
lenet.add(flatten)
lenet.add(dense_1)
lenet.add(dense_2)

# Affichage du résumé du modèle
lenet.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 252, 252, 30)      2280      
                                                                 
 max_pooling2d (MaxPooling2  (None, 126, 126, 30)      0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 124, 124, 16)      4336      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 62, 62, 16)        0         
 g2D)                                                            
                                                                 
 dropout (Dropout)           (None, 62, 62, 16)        0         
                                                                 
 flatten (Flatten)           (None, 61504)            

In [15]:
# Compilation
lenet.compile(loss='sparse_categorical_crossentropy',  # fonction de perte
              optimizer='adam',                 # algorithme de descente de gradient
              metrics=['accuracy'])             # métrique d'évaluation

In [19]:
history_lenet = lenet.fit_generator(train_generator,
                            epochs = 5,
                            validation_data = test_generator,
                           )

C:\Users\yahya\AppData\Local\Temp\ipykernel_12856\2004177776.py:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  history_lenet = lenet.fit_generator(train_generator,


Epoch 1/5
1699/1699 [==============================] - 3446s 2s/step - loss: 3.1065 - accuracy: 0.1217 - val_loss: 3.0803 - val_accuracy: 0.1228
Epoch 2/5
1699/1699 [==============================] - 3212s 2s/step - loss: 3.0903 - accuracy: 0.1230 - val_loss: 3.0821 - val_accuracy: 0.1222
Epoch 3/5
1699/1699 [==============================] - 3174s 2s/step - loss: 3.0871 - accuracy: 0.1235 - val_loss: 3.0733 - val_accuracy: 0.1254
Epoch 4/5
1699/1699 [==============================] - 3173s 2s/step - loss: 3.0810 - accuracy: 0.1249 - val_loss: 3.0617 - val_accuracy: 0.1272
Epoch 5/5
1699/1699 [==============================] - 2155s 1s/step - loss: 3.0837 - accuracy: 0.1252 - val_loss: 3.0779 - val_accuracy: 0.1246


In [20]:
#Prediction - Test set
y_pred_proba = lenet.predict_generator(generator = test_generator)

# l'argmax pour obtenir les classes prédites
y_pred_class = np.argmax(y_pred_proba,axis = 1).astype(int)

# To get classes fro test generator
y_true = test_generator.classes

C:\Users\yahya\AppData\Local\Temp\ipykernel_12856\2314453244.py:2: UserWarning: `Model.predict_generator` is deprecated and will be removed in a future version. Please use `Model.predict`, which supports generators.
  y_pred_proba = lenet.predict_generator(generator = test_generator)


In [21]:
#Get the accuracy score
test_score = lenet.evaluate_generator(test_generator)

print("[INFO] accuracy: {:.2f}%".format(test_score[1] * 100)) 
print("[INFO] Loss: ",test_score[0])

C:\Users\yahya\AppData\Local\Temp\ipykernel_12856\1537030683.py:2: UserWarning: `Model.evaluate_generator` is deprecated and will be removed in a future version. Please use `Model.evaluate`, which supports generators.
  test_score = lenet.evaluate_generator(test_generator)


[INFO] accuracy: 12.46%
[INFO] Loss:  3.077414035797119


In [22]:
print(metrics.classification_report(y_true, y_pred_class))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       126
           1       0.00      0.00      0.00       115
           2       0.00      0.00      0.00       137
           3       0.00      0.00      0.00        32
           4       0.00      0.00      0.00       215
           5       0.00      0.00      0.00        76
           6       0.18      0.01      0.02       179
           7       0.00      0.00      0.00        28
           8       0.00      0.00      0.00       100
           9       0.00      0.00      0.00       120
          10       0.00      0.00      0.00       208
          11       0.00      0.00      0.00       179
          12       0.00      0.00      0.00        35
          13       0.10      0.01      0.02       215
          14       0.00      0.00      0.00        33
          15       0.00      0.00      0.00       176
          16       0.00      0.00      0.00       203
          17       0.00    

C:\Users\yahya\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\yahya\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\yahya\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [24]:
#Enregistrement d'un modèle Keras :
lenet.save("C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\Keras\\lenet_train_img_all_10032024_5_epoch.hdf5")

C:\Users\yahya\anaconda3\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
#Rechargement du modèle :
lenet = load_model("C:\\Users\\yahya\\OneDrive\\Documents\\Rakuten\\Keras\\lenet_train_img_all_10032024_5_epoch.hdf5")